## The BWT Algorithm

![](assets/durbin.png)

# Initialization
- Setup a probability Matrix using the initial probabilities and the first observation.
- Initiate a traceback matrix for path reconstruction 

# Recursion 
- For each position and each possible state, calculate the maximum probability.
- Store both the probabilities and the traceback pointers.
- Apply transition and emission probabilities at each step.

# Termination 
- Identify the final state with the highest probability.
- Traceback through the matrix to reconstruct the optimal path.

$$
\small
\begin{array}{l} 
\quad \textbf{Input: } \\
\quad \quad\quad\;\;\; O = (o_1, \dots, o_T) \text{ the observation sequence} \\
\quad \quad\quad\;\;\; S \text{ the set of hidden states, with } |S| = N \\
\quad \quad\quad\;\;\; \pi(s) \text{ the initial probability of state } s \\
\quad \quad\quad\;\;\; A(s', s) \text{ the transition probability from } s' \text{ to } s \\
\quad \quad\quad\;\;\; B(s, o) \text{ the emission probability of observation } o \text{ in state } s \\
\\
\quad \textbf{Output: } \text{the most likely state sequence } (q_1^*, \dots, q_T^*) \text{ and its probability } P^* \\
\\
\\
\textbf{function } \text{viterbi}(O, S, \pi, A, B) \\
\quad \textbf{for each } \text{state } s \in S \textbf{ do} \\
\quad\quad V_1(s) \gets \pi(s) \cdot B(s, o_1) \\
\quad\quad \text{back}_1(s) \gets \emptyset \\
\quad \textbf{for } t = 2 \textbf{ to } T \textbf{ do} \\
\quad\quad \textbf{for each } \text{state } s \in S \textbf{ do} \\
\quad\quad\quad V_t(s) \gets \max_{s' \in S} \big[ V_{t-1}(s') \cdot A(s', s) \big] \cdot B(s, o_t) \\
\quad\quad\quad \text{back}_t(s) \gets \arg\max_{s' \in S} \big[ V_{t-1}(s') \cdot A(s', s) \big] \\
\quad P^* \gets \max_{s \in S} V_T(s) \\
\quad q_T^* \gets \arg\max_{s \in S} V_T(s) \\
\quad \textbf{for } t = T \textbf{ down to } 2 \textbf{ do} \\
\quad\quad q_{t-1}^* \gets \text{back}_t(q_t^*) \\
\quad \textbf{return } (q_1^*, \dots, q_T^*),\; P^*
\end{array}
$$





$$
\small
\begin{array}{l}
\textbf{function } \text{viterbi}(O, S, \pi, A, B) \\
\quad \textbf{Input: } \text{a sequence of } T \text{ observations } O = (o_1, \dots, o_T) \\
\quad\quad\quad\quad\;\;\, \text{a set of } N \text{ hidden states } S \\
\quad\quad\quad\quad\;\;\, \text{initial probabilities } \pi(s): \text{how likely the sequence starts in state } s \\
\quad\quad\quad\quad\;\;\, \text{transition probabilities } A(s', s): \text{how likely we move from } s' \text{ to } s \\
\quad\quad\quad\quad\;\;\, \text{emission probabilities } B(s, o): \text{how likely state } s \text{ produces observation } o \\
\quad \textbf{Output: } \text{the single most likely sequence of hidden states, and its probability} \\
\\
\quad \textit{// Step 1: Initialization — score every possible starting state} \\
\quad \textbf{for each } \text{state } s \in S \textbf{ do} \\
\quad\quad \text{Set } V_1(s) \text{ to the probability of starting in } s \text{ and emitting } o_1, \text{ i.e. } \pi(s) \cdot B(s, o_1) \\
\quad\quad \text{Record no predecessor for } s \text{ at time } 1 \\
\\
\quad \textit{// Step 2: Recursion — extend the best paths one observation at a time} \\
\quad \textbf{for } t = 2 \textbf{ to } T \textbf{ do} \\
\quad\quad \textbf{for each } \text{state } s \in S \textbf{ do} \\
\quad\quad\quad \text{For every previous state } s', \text{ compute the score of arriving at } s \text{ from } s': \; V_{t-1}(s') \cdot A(s', s) \\
\quad\quad\quad \text{Keep the largest of these scores, multiply it by } B(s, o_t), \text{ and store the result as } V_t(s) \\
\quad\quad\quad \text{Store the } s' \text{ that produced that largest score as } \text{back}_t(s) \text{ (a backpointer)} \\
\\
\quad \textit{// Step 3: Termination — find where the best overall path ends} \\
\quad \text{Let } P^* \text{ be the largest value among } V_T(s) \text{ over all states } s \\
\quad \text{Let } q_T^* \text{ be the state that achieves } P^* \\
\\
\quad \textit{// Step 4: Backtracking — follow the backpointers from the end to the start} \\
\quad \textbf{for } t = T \textbf{ down to } 2 \textbf{ do} \\
\quad\quad \text{Set } q_{t-1}^* \text{ to } \text{back}_t(q_t^*), \text{ the state that led into } q_t^* \\
\\
\quad \textbf{return } \text{the path } (q_1^*, \dots, q_T^*) \text{ and its probability } P^*
\end{array}
$$

CLASS HMM

    CONSTRUCTOR(initial, transitions, emissions)
        derive states from the initial-probability keys
        derive alphabet from the emission-dictionary keys

        validate:
            the state keys are consistent across all dictionaries
            every transition row contains every state
            every emission row contains every alphabet symbol
            all probabilities are finite and between 0 and 1
            initial probabilities sum approximately to 1
            each transition and emission row sums approximately to 1

        store the probabilities
        prepare their logarithms

In [22]:
import numpy as np
import pandas as pd
import math

In [60]:
def _log(probability):
    if probability == 0:
        return -math.inf
    return math.log(probability)

In [61]:
class HiddenMarkov:
    def __init__(self, init_probs, trans_probs, emit_probs):
        self.states = tuple(init_probs)
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs


    def Viterbi(self, observations):
        if not observations:
            return  [], 0.0
        
        alphabet = set()
        
        for emissions in self.emit_probs.values():
            alphabet.update(emissions.keys())

        for symbol in observations:
            if symbol not in alphabet:
                raise ValueError(f"Invalid Observation: {symbol!r}")
        
        # create score table and previous state tables
        scores = []
        previous = []

        for symbol in observations:
            score_column = {}
            previous_column = {}

            for state in self.states:
                score_column[state] = float("-inf")
                previous_column[state] = None
            
            scores.append(score_column)
            previous.append(previous_column)

                # INITIALIZATION
            # FOR each state
            #     score[0][state] =
            #         log_initial[state]
            #         + log_emission[state][observations[0]]
        # Initialization 
        for state in self.states:
            scores[0][state] = (
            _log(self.init_probs[state]) 
            + _log(self.emit_probs[state][observations[0]])
            )
        
        # Recursion 
        # FOR each position t after the first
        #     FOR each current state
        #         FOR each possible preceding state
        #             candidate =
        #                 score[t - 1][preceding state]
        #                 + log_transition[preceding state][current state]
        #                 + log_emission[current state][observations[t]]

        #             IF candidate is better than score[t][current state]
        #                 store candidate as the new score
        #                 store preceding state as the traceback pointer
        for position in range(1, len(observations)):
            for current_state in self.states:
                for preceding_state in self.states:
                    candidate = (
                        scores[position - 1][preceding_state]
                        + _log(self.trans_probs[preceding_state][current_state])
                        + _log(self.emit_probs[current_state][observations[position]])
                    )
                    if candidate > scores[position][current_state]:
                        scores[position][current_state] = candidate
                        previous[position][current_state] = preceding_state
    #test 1 
    #     print("First observation:", observations[0])

    #     for state in self.states:
    #         print(
    #             state,
    #             "score:", scores[0][state],
    #             "previous:", previous[0][state]
    # )
    
    #test 2
        for position, symbol in enumerate(observations):
            print(f"Position {position}, observation {symbol}")

            for state in self.states:
                print(
                    f"  State {state}: "
                    f"score={scores[position][state]:.6f}, "
                    f"previous={previous[position][state]}"
                    )
    # TERMINATION
    #   choose the state with the greatest score at the last position
    #   save that score as the best log probability

    #   IF that score is negative infinity
    #       REPORT that no valid path can produce the observations
        final = max(scores[-1], key = scores[-1].get)
        best_log = scores[-1][final]
        print("Best final state:", final)
        print("Best log probability:", best_log)

        if best_log == -math.inf:
            raise ValueError("No Valid path can produce the observations.")
        
        #     TRACEBACK
        # put the chosen final state at the end of the path
        # follow previous-state pointers backward
        # RETURN the path and best log probability
        path = [final]
        
        for position in range(len(observations) - 1, 0, -1):
            preceding_state = previous[position][path[-1]]
            path.append(preceding_state)

        path.reverse()

        print(path, best_log)

    


In [62]:
model = HiddenMarkov(init_probs, trans_probs, emit_probs)
model.Viterbi(obs)

Position 0, observation G
  State I: score=-2.525729, previous=None
  State G: score=-1.832581, previous=None
Position 1, observation G
  State I: score=-3.798694, previous=I
  State G: score=-3.547380, previous=G
Position 2, observation C
  State I: score=-5.071660, previous=I
  State G: score=-5.262178, previous=G
Position 3, observation A
  State I: score=-7.730920, previous=I
  State G: score=-6.571512, previous=G
Position 4, observation C
  State I: score=-9.003886, previous=I
  State G: score=-8.286310, previous=G
Position 5, observation T
  State I: score=-11.663146, previous=I
  State G: score=-9.595643, previous=G
Position 6, observation G
  State I: score=-12.814519, previous=G
  State G: score=-11.310442, previous=G
Position 7, observation A
  State I: score=-15.473779, previous=I
  State G: score=-12.619775, previous=G
Position 8, observation A
  State I: score=-17.224945, previous=G
  State G: score=-13.929108, previous=G
Best final state: G
Best log probability: -13.92910

SAFE_LOG(probability):

    IF probability is zero
        RETURN negative infinity
    RETURN log(probability)

In [ ]:
def safe_log(probability):
    if probability == 0:
        return -math.inf
    return math.log(probability)

METHOD viterbi(observations)

    IF observations is empty
        RETURN empty path, log probability 0

    IF any observation is outside the model's alphabet
        REPORT an invalid observation

    create score table, initially negative infinity
    create previous-state table, initially NONE

    INITIALIZATION
        FOR each state
            score[0][state] =
                log_initial[state]
                + log_emission[state][observations[0]]

    RECURSION
        FOR each position t after the first
            FOR each current state
                FOR each possible preceding state
                    candidate =
                        score[t - 1][preceding state]
                        + log_transition[preceding state][current state]
                        + log_emission[current state][observations[t]]

                    IF candidate is better than score[t][current state]
                        store candidate as the new score
                        store preceding state as the traceback pointer

    TERMINATION
        choose the state with the greatest score at the last position
        save that score as the best log probability

        IF that score is negative infinity
            REPORT that no valid path can produce the observations

    TRACEBACK
        put the chosen final state at the end of the path
        follow previous-state pointers backward
        RETURN the path and best log probability

In [24]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [26]:
model = HiddenMarkov(init_probs, trans_probs, emit_probs)
model.Viterbi(obs)

Position 0, observation G
  State I: score=-2.525729, previous=None
  State G: score=-1.832581, previous=None
Position 1, observation G
  State I: score=-3.798694, previous=I
  State G: score=-3.547380, previous=G
Position 2, observation C
  State I: score=-5.071660, previous=I
  State G: score=-5.262178, previous=G
Position 3, observation A
  State I: score=-7.730920, previous=I
  State G: score=-6.571512, previous=G
Position 4, observation C
  State I: score=-9.003886, previous=I
  State G: score=-8.286310, previous=G
Position 5, observation T
  State I: score=-11.663146, previous=I
  State G: score=-9.595643, previous=G
Position 6, observation G
  State I: score=-12.814519, previous=G
  State G: score=-11.310442, previous=G
Position 7, observation A
  State I: score=-15.473779, previous=I
  State G: score=-12.619775, previous=G
Position 8, observation A
  State I: score=-17.224945, previous=G
  State G: score=-13.929108, previous=G
Best final state: G
Best log probability: -13.92910

![](assets/rotation.png)

![](assets/rotation.png)